# 7. Autoresearch-agent memory-debug loop

This is the case study requested by the proposal: an autonomous research agent generates ideas, retrieves prior experiment memories, and is debugged.

## Proposal Part 4 case study mapping

The proposal's Part 4 case study describes single-GPU/autonomous research with:

- 100+ overnight experiments.
- Working memory over recent experiments.
- Episodic compressed memory.
- Semantic/LLM-extracted cached memory.
- Fewer redundant experiments, faster convergence, and reduced API cost.

This notebook implements the tutorial-scale analogue: real benchmark outcomes become experiment memories, the agent proposes new ideas, retrieves relevant prior results, then decides **keep**, **revise**, or **discard** while diagnosing memory failures.

In [1]:
import os
# Optional OpenAI-compatible idea generation. The default executed path remains offline.
# To use the API, set OPENAI_API_KEY outside the notebook and run experiment/run.py with:
# --backend openai-compatible --base-url http://127.0.0.1:8317/api/provider/codex/v1 --model gpt-5.5
print("OPENAI_API_KEY present:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY present: False


In [2]:
from pathlib import Path
import json
import os
import sys

# Colab guidance:
# 1. Upload or clone this repository into the Colab runtime.
# 2. Set PROJECT_ROOT to the repository root if auto-detection fails.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiment" / "run.py").exists():
    candidates = [Path("/content/kdd"), Path("/content/drive/MyDrive/kdd"), Path("/Users/syaikhipin/Documents/Phd/Publish Paper/kdd")]
    PROJECT_ROOT = next((p for p in candidates if (p / "experiment" / "run.py").exists()), PROJECT_ROOT)

EXPERIMENT_DIR = PROJECT_ROOT / "experiment"
RESULTS_DIR = EXPERIMENT_DIR / "results"
sys.path.insert(0, str(EXPERIMENT_DIR))
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Experiment code exists:", (EXPERIMENT_DIR / "run.py").exists())

PROJECT_ROOT = /Users/syaikhipin/Documents/Phd/Publish Paper/kdd
Experiment code exists: True


In [3]:
import subprocess
latest_real_metrics = sorted(RESULTS_DIR.glob("run_*_real_metrics.json"))[-1]
cmd = [
    sys.executable, str(EXPERIMENT_DIR / "run.py"),
    "--mode", "autoresearch-agent",
    "--backend", "offline",
    "--benchmark-metrics", str(latest_real_metrics),
    "--use-cases", "locomo", "autoresearch", "hpo", "memoryarena", "longmemeval", "lcbench",
    "--ideas-per-case", "2",
    "--top-k", "5",
    "--eval-backend", "offline",
    "--eval-limit", "12",
]
print(" ".join(map(str, cmd)))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=120)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0

/Users/syaikhipin/anaconda3/bin/python /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/run.py --mode autoresearch-agent --backend offline --benchmark-metrics /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/results/run_20260509_002743_real_metrics.json --use-cases locomo autoresearch hpo memoryarena longmemeval lcbench --ideas-per-case 2 --top-k 5 --eval-backend offline --eval-limit 12
metric	value
ideas	12
kept	2
revised	4
discarded	6
mean_retrieval_precision	0.2333
mean_retrieval_recall	0.2708
memory_utilization_rate	0.3333
redundant_idea_rate	0.5
mean_novelty_score	0.3432
mean_semantic_score	0.2602
semantic_pass_rate	0.3333
failure_modes	{"none": 2, "redundant_idea": 6, "retrieval_miss": 4}
decisions	{"discard": 6, "keep": 2, "revise": 4}
Autoresearch agent report: /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/results/run_20260509_003040_autoresearch_agent_report.md

Raw trace: /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/resul

## Semantic evaluation of generated ideas

Notebook 7 now evaluates generated idea records with the offline semantic evaluator. Rhesis and Semantica can use the same `--eval-backend` flag for small maintainer smoke tests outside the live tutorial path.

In [4]:
latest = sorted(RESULTS_DIR.glob("run_*_autoresearch_agent_raw.json"))[-1]
print("Reading", latest)
trace = json.loads(latest.read_text())
for r in trace["records"]:
    print(r["step"], r["idea_id"], r["dataset"], r["decision"], r["failure_category"], "novelty=", r["novelty_score"], "signal=", r["observed_signal"])

Reading /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/results/run_20260509_003040_autoresearch_agent_raw.json
1 locomo-1 LoCoMo keep none novelty= 1.0 signal= 1.0
2 locomo-2 LoCoMo discard redundant_idea novelty= 0.4472 signal= 1.0
3 autoresearch-1 autoresearch traces keep none novelty= 0.5515 signal= 1.0
4 autoresearch-2 autoresearch traces discard redundant_idea novelty= 0.0286 signal= 1.0
5 hpo-1 HPOBench or LCBench revise retrieval_miss novelty= 0.5333 signal= 0.0
6 hpo-2 HPOBench or LCBench discard redundant_idea novelty= 0.0297 signal= 0.0
7 memoryarena-1 MemoryArena revise retrieval_miss novelty= 0.4961 signal= 0.0
8 memoryarena-2 MemoryArena discard redundant_idea novelty= 0.0319 signal= 0.0
9 longmemeval-1 LongMemEval revise retrieval_miss novelty= 0.4615 signal= 0.0
10 longmemeval-2 LongMemEval discard redundant_idea novelty= 0.0341 signal= 0.0
11 lcbench-1 LCBench revise retrieval_miss novelty= 0.4733 signal= 0.0
12 lcbench-2 LCBench discard redundant_idea nov

Guidance: use this for the Part 4 case study. Discuss why the agent keeps some ideas, revises unsupported ones, and discards redundant ideas.